# Hierarchical Risk Parity (HRP)

**Objective key:** `hrp` &nbsp;·&nbsp; **Family:** Classical &nbsp;·&nbsp; **Speed:** Fast

## Algorithm

Three-step procedure:
1. **Tree Clustering** — build a hierarchical clustering of assets based on their correlation distance.
2. **Quasi-Diagonalisation** — reorder the covariance matrix so that similar assets are adjacent.
3. **Recursive Bisection** — allocate risk weights top-down: each sub-cluster receives weight  
   inversely proportional to its variance.

HRP does **not** invert the covariance matrix, making it robust when Σ is rank-deficient  
or when the number of assets approaches the number of observations.

## Papers

- **Foundational:** López de Prado (2016) — *Building Diversified Portfolios that Outperform Out-of-Sample*  
  https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678
- **Modern:** López de Prado (2018) — *Advances in Financial Machine Learning* Ch. 16  
  https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086

## Assumptions

- `mu` is passed but only used to compute `OptimizationResult` metrics; HRP does not use it in weight allocation.
- `Sigma` is annualised. Weights are not bounded by min/max (HRP bypasses convex constraint).
- `seed=42` (reproducibility of synthetic data only).

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 12
mu = rng.uniform(0.06, 0.20, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"{n} assets, Sigma shape {Sigma.shape}")

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="hrp",
    asset_names=asset_names,
    seed=42,
)

print(f"\nObjective:       {result.objective}")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print(f"Weights sum:     {result.weights.sum():.6f}")
print("\nWeights (sorted descending):")
idx_sorted = result.weights.argsort()[::-1]
for i in idx_sorted:
    print(f"  {asset_names[i]}: {result.weights[i]:.4f}")

In [ ]:
# HRP is fully diversified — all weights should be positive
assert all(result.weights > 0), "HRP should assign positive weight to all assets"
print("All weights > 0 ✓")

# Compare concentration (max weight) vs equal weight
ew_max = 1.0 / n
hrp_max = result.weights.max()
print(f"Max weight — HRP: {hrp_max:.4f}  |  Equal weight: {ew_max:.4f}")